1. Load and Inspect CSV file


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 1. Load your existing synthetic CSV file
# Ensure 'cloud_cost_poc_data.csv' is in the same directory as your notebook
df = pd.read_csv("data/cloud_cost_poc_data.csv")

# 2. Parse timestamps and extract basic time components
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Hour"] = df["Timestamp"].dt.hour

print(f"Successfully loaded dataset with {len(df)} rows.")
df.head()

Successfully loaded dataset with 17288 rows.


,Timestamp,Service,Region,Hour,Usage_Amount,Cost
0,2026-03-01,EC2,us-east-1,0,11.968397,2.125275
1,2026-03-01,EC2,eu-west-1,0,7.472324,0.658540
2,2026-03-01,S3,us-east-1,0,6.911418,0.833747
3,2026-03-01,S3,eu-west-1,0,6.911511,1.124711
4,2026-03-01,RDS,us-east-1,0,23.248572,1.467165


2. Add MLFlow

In [2]:
import os
import mlflow
import mlflow.sklearn

# Set up local MLflow directories
db_path = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("Cloud_Cost_Anomaly_Engine")

<Experiment: artifact_location='./mlruns/1', experiment_id='1', lifecycle_stage='active', name='Cloud_Cost_Anomaly_Engine', tags={}>

2. Feature engineering and One-hot encoding

In [3]:
# 1. Transform categorical columns into a machine-readable format
df_features = pd.get_dummies(df, columns=["Service", "Region"], drop_first=False)

# 2. Filter for numerical features only
# We exclude columns like Timestamp to force the model to evaluate usage vs cost relationships
feature_cols = [col for col in df_features.columns if col not in ["Timestamp", "Hour"]]
X = df_features[feature_cols]

# 3. Scale values to ensure large usage metrics don't skew the distance mathematical calculations
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features ready for training. Total feature dimensions: {X_scaled.shape[1]}")


Features ready for training. Total feature dimensions: 8


3. Train Isolation forest

In [6]:
import logging

# Silence MLflow's environment logging warnings
logging.getLogger("mlflow.utils.environment").setLevel(logging.ERROR)
# Configure hyperparameters
n_est = 100
contam = 0.015
rand_st = 42

with mlflow.start_run() as run:
    # 1. Train Model
    iso_forest = IsolationForest(n_estimators=n_est, contamination=contam, random_state=rand_st)
    df["Anomaly_Label"] = iso_forest.fit_predict(X_scaled)
    df["Anomaly_Score"] = iso_forest.decision_function(X_scaled)
    df["Is_Anomaly"] = df["Anomaly_Label"].map({1: False, -1: True})
    
    # 2. Compute Performance Validation Metrics
    total_anomalies = int(df["Is_Anomaly"].sum())
    max_anomaly_cost = float(df[df["Is_Anomaly"] == True]["Cost"].max())
    anomaly_ratio = float(total_anomalies / len(df))
    
    print(f"Training session complete. Detected {total_anomalies} anomalies.")

    # 3. Log Parameters and Metrics to Local MLflow DB
    mlflow.log_param("n_estimators", n_est)
    mlflow.log_param("contamination", contam)
    mlflow.log_param("random_state", rand_st)
    mlflow.log_param("num_features", X_scaled.shape[1])
    
    mlflow.log_metric("total_anomalies_detected", total_anomalies)
    mlflow.log_metric("anomaly_ratio", anomaly_ratio)
    mlflow.log_metric("max_anomaly_cost_flagged", max_anomaly_cost)
    
    # 4. Generate & Save Visualization Directly into the MLflow Run
    plt.figure(figsize=(14, 6))
    plt.scatter(df[df["Is_Anomaly"] == False]["Timestamp"], df[df["Is_Anomaly"] == False]["Cost"], c="#3498db", label="Normal", alpha=0.4, s=12)
    plt.scatter(df[df["Is_Anomaly"] == True]["Timestamp"], df[df["Is_Anomaly"] == True]["Cost"], c="#e74c3c", label="Anomaly", alpha=0.9, s=35)
    plt.title("Cloud Cost Optimization Engine (Tracked Run)")
    plt.legend()
    plt.savefig("poc_anomaly_chart.png")
    plt.close()
    
    mlflow.log_artifact("poc_anomaly_chart.png")  # Saves visual plot to local artifact registry
    
    # 5. Log and Bundle the Model and Scaler together
    artifacts = {"scaler": "cloud_cost_scaler.pkl"}
    with open("cloud_cost_scaler.pkl", "wb") as f:
        import pickle
        pickle.dump(scaler, f)
        
    # Register the model in your local catalog registry
    mlflow.sklearn.log_model(
        sk_model=iso_forest,
        artifact_path="model",
        registered_model_name="Cloud_Cost_Optimizer_Model"
    )
    
    run_id = run.info.run_id
    print(f"Successfully logged run {run_id} to local registry.")


Training session complete. Detected 260 anomalies.


Registered model 'Cloud_Cost_Optimizer_Model' already exists. Creating a new version of this model...
2026/06/25 12:22:08 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: Cloud_Cost_Optimizer_Model, version 3


Successfully logged run 9f9c16686db44bbd86401cbc8fa0439a to local registry.


Created version '3' of model 'Cloud_Cost_Optimizer_Model'.


4. Automated testing, validation and guardrails

In [8]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_name = "Cloud_Cost_Optimizer_Model"


# 1. Fetch ALL registered versions for this model to prevent empty list crashes
all_versions = client.search_model_versions(f"name='{model_name}'")
if not all_versions:
    raise ValueError(f"No versions found for model name '{model_name}'. Run the training cell first.")
# 2. Extract the absolute newest version number dynamically
# MLflow returns search results ordered by creation time, or we can take the max version int
latest_version_num = max(int(v.version) for v in all_versions)
print(f"Evaluating Model Version: {latest_version_num}")

# --- AUTOMATED TEST & VALIDATION GATE ---
# Rule 1: The model should not tag more than 5% of our traffic as anomalous (Prevents spam alerts)
# Rule 2: The model must successfully isolate severe baseline leaks over $500
passed_validation = True

if anomaly_ratio > 0.05:
    print(f"[FAIL] Degradation Warning: Candidate flags too many points ({anomaly_ratio:.2%}).")
    passed_validation = False

if max_anomaly_cost < 50:
    print("[FAIL] Degradation Warning: Candidate missed heavy structural spend leaks.")
    passed_validation = False

# --- AUTOMATED VERSION MANAGMENT (PROMOTION) ---
if passed_validation:
    print(f"[PASS] Performance checks verified. Promoting Version {latest_version_num} to Production!")
    # Mark this version as production active
    client.transition_model_version_stage(
        name=model_name,
        version=latest_version_num,
        stage="Production",
        archive_existing_versions=True # Automatically demotes the old model to "Archived"
    )
else:
    print(f"[REJECTED] Version {latest_version_num} degraded parameters. Demoting to Staging/Archived.")
    client.transition_model_version_stage(
        name=model_name,
        version=latest_version_num,
        stage="Archived"
    )


Evaluating Model Version: 3
[PASS] Performance checks verified. Promoting Version 3 to Production!
